# 93 — Scoring

Loads the pre-normalized layer stack written by `92_normalization.ipynb` and combines it using `weights.yaml` into a single livability score. Normalization is no longer done here — the percentile transform in `92_normalization.ipynb` is the only place layers are rescaled.

In [ ]:
import xarray as xr

from common import PROCESSED_DIR, load_weights, save_variable, weighted_score

## Weights

Raw values from `weights.yaml`, zeros dropped. Renormalization happens after we know which layers actually loaded.

In [ ]:
weights = load_weights()
weights

## Load normalized layers

Read the percentile-transformed stack from `processed/normalized.nc`. Only variables that are both in the file and have a non-zero weight are kept, so scoring still works incrementally as more variables are added — a layer that isn't in `normalized.nc` yet is skipped with a warning.

In [ ]:
ds = xr.open_dataset(PROCESSED_DIR / 'normalized.nc')
normed = {}
for name in weights:
    if name in ds.data_vars:
        normed[name] = ds[name]
    else:
        print(f'skip {name}: not in normalized.nc')
print(f'loaded {len(normed)} of {len(weights)} layers')

## Weighted sum

Per-cell weighted average. If a layer is missing at a cell, its weight is redistributed across the layers that do have data there — so a missing variable neither drags the score down nor discards the cell. A cell is only NaN where every layer is missing.

In [ ]:
score = weighted_score(normed, weights)
score.name = 'livability'

## Save

In [ ]:
out = save_variable(score, 'score')
print(f'wrote {out}')